# Keyword Stems (Downstream Branch B)

Improved extraction using **stemming** (SnowballStemmer) to consolidate word variants (color/colored/colorful → color),
with filtering of numbers, non-alpha tokens, and short fragments.

### Improvements over v1
- Stems instead of lemmas to merge word variants (e.g., confuse/confused/confusing → confus)
- Filters out numbers (`1`, `1/3`, `15`), non-alpha tokens (`2d/3d`, `couldnt\`see`)
- Keeps a readable representative word for each stem (the shortest word mapping to that stem)

### Input
- `data_processing/image_phrase_matching.csv` — produced by `5_keyword_extraction.ipynb`
- Optional lineage section: `data_processing/phrase_lineage_all_vistypes.csv`, with `original_phrase` and `final_subtopic_phrase`. Supply this separately; current notebook 3 cells do not produce it.

### Output
- `data_processing/unique_stems.csv` — all-phrases stem extraction
- `data_processing/unique_stems_by_topic.csv` — per-topic with image counts
- `data_processing/unique_stems_by_vistype.csv` — per-VisType with image counts

### Execution guide

**Required:** run cells 2-7 for the stem tables.

**Optional lineage:** run cells 9-11 for `Original -> HumanCurated -> FinalPhrase` and its topic/VisType summaries only when the additional lineage CSV is available.

In [ ]:
# --- portable setup (added by clean-up) ---
import os, sys
REPO_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
os.makedirs('data_processing', exist_ok=True)
from data_loader import load_sheet, load_paired_comparison
import config


In [ ]:
import pandas as pd
import spacy
import re
from collections import Counter, defaultdict
from nltk.stem import SnowballStemmer

nlp = spacy.load('en_core_web_sm')
stemmer = SnowballStemmer('english')

# Load image-based phrase matching (produced by KeywordExtraction.ipynb)
df_img = pd.read_csv('data_processing/image_phrase_matching.csv')
print(f'Loaded image_phrase_matching.csv: {len(df_img)} records, {df_img["ImageName"].nunique()} images')

topic_cols = [
    'Immediacy / Cognitive Load',
    'Data Density / Image Clutter',
    'Visual Encoding Clarity',
    'Semantics / Text Legibility',
    'Schema',
    'Color, Symbol, and Texture Details',
    'Aesthetics Uncertainty',
]
CANONICAL_VISTYPES = ['Area', 'Bar', 'Cont.-ColorPatn', 'Glyph', 'Grid', 'Line', 'Node-link', 'Point', 'Text']

In [ ]:
# ── Helper functions ────────────────────────────────────────────────────────────

WHITELIST_TOKENS = {'2d', '3d', '2d/3d'}

def is_valid_token(token):
    """Check if a spaCy token is a valid content word (not stop/punct/space/number)."""
    text = token.text.lower()
    if text in WHITELIST_TOKENS:
        return True
    if token.is_stop or token.is_punct or token.is_space:
        return False
    if token.pos_ == 'NUM':
        return False
    # Must be purely alphabetic (allows hyphens inside words)
    if not re.match(r'^[a-z]+(-[a-z]+)*$', text):
        return False
    if len(text) <= 1:
        return False
    return True

def extract_stems_from_phrase(phrase):
    """Extract set of (stem, representative_word) from a single phrase."""
    if not isinstance(phrase, str) or phrase.strip() in ('', '-'):
        return set()
    doc = nlp(phrase)
    results = set()
    for token in doc:
        if not is_valid_token(token):
            continue
        word = token.lemma_.lower()
        stem = stemmer.stem(word)
        results.add((stem, word))
    return results

# Build a global stem → best representative word mapping
# Use the shortest lemma that maps to each stem
all_phrases = pd.concat([
    df_img['Original'], df_img['AIcurated'], df_img['HumanCurated']
]).dropna().unique()

stem_to_word = {}
for phrase in all_phrases:
    for stem, word in extract_stems_from_phrase(phrase):
        if stem not in stem_to_word or len(word) < len(stem_to_word[stem]):
            stem_to_word[stem] = word

def get_stems_from_phrase(phrase):
    """Extract set of stems from a phrase."""
    return {stem for stem, _ in extract_stems_from_phrase(phrase)}

print(f'Total unique stems: {len(stem_to_word)}')
# Show some examples of consolidation
examples = ['color', 'confus', 'clear', 'clutter', 'annot', 'complex', 'bright']
for s in examples:
    if s in stem_to_word:
        print(f'  stem "{s}" → "{stem_to_word[s]}"')

In [ ]:
# ── All-phrases unique stems with POS ──────────────────────────────────────────
def extract_stem_pos_pairs(phrases):
    """Extract unique (stem, POS) pairs from phrases, returning (representative_word, POS)."""
    stem_pos = set()
    for phrase in phrases:
        if not isinstance(phrase, str) or phrase.strip() in ('', '-'):
            continue
        doc = nlp(phrase)
        for token in doc:
            if not is_valid_token(token):
                continue
            word = token.lemma_.lower()
            stem = stemmer.stem(word)
            stem_pos.add((stem, token.pos_))
    # Convert stems to representative words
    return sorted((stem_to_word.get(s, s), pos) for s, pos in stem_pos)

cols = {'Original': 'Original', 'AIcurated': 'AIcurated', 'HumanCurated': 'HumanCurated'}
word_lists = {}
for col, label in cols.items():
    phrases = df_img[col].dropna()
    phrases = phrases[phrases.str.strip() != ''].unique()
    wp = extract_stem_pos_pairs(phrases)
    word_lists[label] = wp
    print(f'{label}: {len(phrases)} phrases → {len(wp)} unique stems')

max_len = max(len(v) for v in word_lists.values())
frames = []
for label in cols.values():
    wp = word_lists[label]
    d = pd.DataFrame(wp, columns=[f'{label}Word', f'{label}POS'])
    d = d.reindex(range(max_len)).fillna('')
    frames.append(d)

df_out = pd.concat(frames, axis=1)
csv_path = 'data_processing/unique_stems.csv'
df_out.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({max_len} rows)')
df_out.head(15)

In [ ]:
# ── Per-topic unique stems with image counts ──────────────────────────────────
frames = []
for topic in topic_cols:
    subset = df_img[df_img['Topic'] == topic]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('AIcurated', 'AIcurated'), ('HumanCurated', 'HumanCurated')]:
        # Count images per stem
        stem_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_stems = set()
            for phrase in phrases:
                img_stems |= get_stems_from_phrase(phrase)
            for stem in img_stems:
                stem_images[stem] += 1

        sorted_items = sorted(stem_images.items())
        col_data[f'{topic}\n{label}'] = [stem_to_word.get(s, s) for s, _ in sorted_items]
        col_data[f'{topic}\n{label} Count'] = [str(c) for _, c in sorted_items]

    if col_data:
        ml = max(len(v) for v in col_data.values())
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))
    frames.append(pd.DataFrame(col_data))

    n_orig = len([x for x in col_data[f'{topic}\nOriginal'] if x])
    n_ai   = len([x for x in col_data[f'{topic}\nAIcurated'] if x])
    n_hc   = len([x for x in col_data[f'{topic}\nHumanCurated'] if x])
    print(f'{topic}: Orig={n_orig}, AI={n_ai}, HC={n_hc}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_topics = pd.concat(frames, axis=1)

csv_path = 'data_processing/unique_stems_by_topic.csv'
df_topics.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_topics)} rows, {len(df_topics.columns)} columns)')
df_topics.head(10)

In [ ]:
# ── Per-VisType unique stems with image counts ────────────────────────────────
frames = []
for vt in CANONICAL_VISTYPES:
    subset = df_img[df_img['VisType'] == vt]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('AIcurated', 'AIcurated'), ('HumanCurated', 'HumanCurated')]:
        stem_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_stems = set()
            for phrase in phrases:
                img_stems |= get_stems_from_phrase(phrase)
            for stem in img_stems:
                stem_images[stem] += 1

        sorted_items = sorted(stem_images.items())
        col_data[f'{vt}\n{label}'] = [stem_to_word.get(s, s) for s, _ in sorted_items]
        col_data[f'{vt}\n{label} Count'] = [str(c) for _, c in sorted_items]

    if col_data:
        ml = max(len(v) for v in col_data.values())
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))

    frames.append(pd.DataFrame(col_data))
    n_orig = len([x for x in col_data[f'{vt}\nOriginal'] if x])
    n_ai   = len([x for x in col_data[f'{vt}\nAIcurated'] if x])
    n_hc   = len([x for x in col_data[f'{vt}\nHumanCurated'] if x])
    print(f'{vt}: Orig={n_orig}, AI={n_ai}, HC={n_hc}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_vistype = pd.concat(frames, axis=1)

csv_path = 'data_processing/unique_stems_by_vistype.csv'
df_vistype.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_vistype)} rows, {len(df_vistype.columns)} columns)')
df_vistype.head(10)

## Word Reduction Pipeline: Original → HumanCurated → FinalPhrase

Using the separately supplied `data_processing/phrase_lineage_all_vistypes.csv`, map each image's HumanCurated phrase
to its final shortlisted phrase (if any), then extract unique stems at each level to show the
progressive word reduction: raw annotator text → human-curated phrases → final sub-topic phrases.

In [ ]:
# ── Load final phrase mapping from PhraseReductionPipeline ─────────────────────
df_lineage = pd.read_csv('data_processing/phrase_lineage_all_vistypes.csv')

# Build unique mapping: HumanCurated phrase → FinalPhrase
# (original_phrase in lineage = HumanCurated in image_phrase_matching)
hc_to_final = {}
for _, row in df_lineage.iterrows():
    hc = row['original_phrase']
    final = row['final_subtopic_phrase']
    if isinstance(final, str) and final.strip():
        hc_to_final[hc] = final
    elif hc not in hc_to_final:
        hc_to_final[hc] = ''

# Add FinalPhrase column to df_img
df_img['FinalPhrase'] = df_img['HumanCurated'].map(hc_to_final).fillna('')

n_with_final = (df_img['FinalPhrase'] != '').sum()
n_unique_final = df_img[df_img['FinalPhrase'] != '']['FinalPhrase'].nunique()
print(f'Mapped {n_with_final}/{len(df_img)} records to a final phrase ({n_unique_final} unique final phrases)')
print(f'\nFinal phrases: {sorted(df_img[df_img["FinalPhrase"] != ""]["FinalPhrase"].unique())}')

# ── All-phrases unique stems: Original → HumanCurated → FinalPhrase ───────────
pipeline_cols = {'Original': 'Original', 'HumanCurated': 'HumanCurated', 'FinalPhrase': 'FinalPhrase'}
word_lists_pipeline = {}
for col, label in pipeline_cols.items():
    phrases = df_img[col].dropna()
    phrases = phrases[phrases.str.strip() != ''].unique()
    wp = extract_stem_pos_pairs(phrases)
    word_lists_pipeline[label] = wp
    print(f'{label}: {len(phrases)} phrases → {len(wp)} unique stems')

max_len = max(len(v) for v in word_lists_pipeline.values())
frames = []
for label in pipeline_cols.values():
    wp = word_lists_pipeline[label]
    d = pd.DataFrame(wp, columns=[f'{label}Word', f'{label}POS'])
    d = d.reindex(range(max_len)).fillna('')
    frames.append(d)

df_pipeline = pd.concat(frames, axis=1)
csv_path = 'data_processing/unique_stems_pipeline.csv'
df_pipeline.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({max_len} rows)')
df_pipeline.head(15)

In [ ]:
# ── Per-topic unique stems: Original → HumanCurated → FinalPhrase ──────────────
frames = []
for topic in topic_cols:
    subset = df_img[df_img['Topic'] == topic]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('HumanCurated', 'HumanCurated'), ('FinalPhrase', 'FinalPhrase')]:
        stem_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_stems = set()
            for phrase in phrases:
                if phrase.strip() == '':
                    continue
                img_stems |= get_stems_from_phrase(phrase)
            for stem in img_stems:
                stem_images[stem] += 1

        sorted_items = sorted(stem_images.items())
        col_data[f'{topic}\n{label}'] = [stem_to_word.get(s, s) for s, _ in sorted_items]
        col_data[f'{topic}\n{label} Count'] = [str(c) for _, c in sorted_items]

    if col_data:
        ml = max(len(v) for v in col_data.values()) if any(col_data.values()) else 0
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))
    frames.append(pd.DataFrame(col_data))

    n_orig = len([x for x in col_data.get(f'{topic}\nOriginal', []) if x])
    n_hc   = len([x for x in col_data.get(f'{topic}\nHumanCurated', []) if x])
    n_fp   = len([x for x in col_data.get(f'{topic}\nFinalPhrase', []) if x])
    print(f'{topic}: Orig={n_orig}, HC={n_hc}, Final={n_fp}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_topics_pipeline = pd.concat(frames, axis=1)

csv_path = 'data_processing/unique_stems_by_topic_pipeline.csv'
df_topics_pipeline.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_topics_pipeline)} rows, {len(df_topics_pipeline.columns)} columns)')
df_topics_pipeline.head(10)

In [ ]:
# ── Per-VisType unique stems: Original → HumanCurated → FinalPhrase ────────────
frames = []
for vt in CANONICAL_VISTYPES:
    subset = df_img[df_img['VisType'] == vt]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('HumanCurated', 'HumanCurated'), ('FinalPhrase', 'FinalPhrase')]:
        stem_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_stems = set()
            for phrase in phrases:
                if phrase.strip() == '':
                    continue
                img_stems |= get_stems_from_phrase(phrase)
            for stem in img_stems:
                stem_images[stem] += 1

        sorted_items = sorted(stem_images.items())
        col_data[f'{vt}\n{label}'] = [stem_to_word.get(s, s) for s, _ in sorted_items]
        col_data[f'{vt}\n{label} Count'] = [str(c) for _, c in sorted_items]

    if col_data:
        ml = max(len(v) for v in col_data.values()) if any(col_data.values()) else 0
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))

    frames.append(pd.DataFrame(col_data))
    n_orig = len([x for x in col_data.get(f'{vt}\nOriginal', []) if x])
    n_hc   = len([x for x in col_data.get(f'{vt}\nHumanCurated', []) if x])
    n_fp   = len([x for x in col_data.get(f'{vt}\nFinalPhrase', []) if x])
    print(f'{vt}: Orig={n_orig}, HC={n_hc}, Final={n_fp}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_vistype_pipeline = pd.concat(frames, axis=1)

csv_path = 'data_processing/unique_stems_by_vistype_pipeline.csv'
df_vistype_pipeline.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_vistype_pipeline)} rows, {len(df_vistype_pipeline.columns)} columns)')
df_vistype_pipeline.head(10)